# IFCE Campus Tauá - Tecnologia em Telemática
## Banco de Dados (2026.2) — Aula 06: Entidades Fracas e Modelo EER
**Prof. Me. Reginaldo Fernandes**

---
### Objetivos Práticos Deste Caderno:
1. **Entidades Fracas e Discriminadores:** Compreender a dependência existencial e de identificação entre uma entidade forte proprietária (`DISPOSITIVO_REDE`) e sua entidade fraca dependente (`PORTA_INTERFACE`).
2. **Chave Primária Composta por Dependência:** Demonstrar como a chave primária da entidade proprietária unida ao discriminador forma a identificação global unívoca `(ip_gestao, numero_porta)`.
3. **Modelo EER (Especialização/Generalização):** Modelar hierarquias de herança onde subclasses (`Roteador`, `Switch`) herdam propriedades da superclasse (`DispositivoGenerico`).
4. **Restrições de Disjunção ($d$) e Integralidade Total:** Auditar e interceptar violações de regras semânticas em memória.

### 1. Definição das Classes para Entidades Fracas e Identificadores

In [ ]:
from enum import Enum
from typing import Dict, Any, Optional, Set, List

class EntidadeForte:
    """Entidade com chave primária autônoma (ip_gestao)."""
    def __init__(self, ip_gestao: str, hostname: str, localizacao: str):
        self.ip_gestao = ip_gestao
        self.hostname = hostname
        self.localizacao = localizacao
        self.portas_associadas: Dict[str, 'PortaInterface'] = {}

    def __repr__(self) -> str:
        return f"<Dispositivo ip={self.ip_gestao} host={self.hostname}>"

class PortaInterface:
    """Entidade Fraca: identificada pelo discriminador numero_porta + proprietario."""
    def __init__(self, numero_porta: str, tipo_midia: str, velocidade_mbps: int):
        self.numero_porta = numero_porta  # Chave Parcial / Discriminador
        self.tipo_midia = tipo_midia
        self.velocidade_mbps = velocidade_mbps
        self.dispositivo_proprietario: Optional[EntidadeForte] = None

    @property
    def chave_completa(self) -> Optional[tuple]:
        if not self.dispositivo_proprietario:
            return None
        return (self.dispositivo_proprietario.ip_gestao, self.numero_porta)

    def __repr__(self) -> str:
        chave = self.chave_completa or "SEM_PROPRIETARIO"
        return f"<Porta {chave} ({self.tipo_midia}, {self.velocidade_mbps}Mbps)>"

class RelacionamentoIdentificador:
    """Garante integridade existencial e unicidade do discriminador."""
    def __init__(self, nome: str = "CONTEM_PORTA"):
        self.nome = nome
        self.ligacoes: Dict[tuple, PortaInterface] = {}

    def vincular_porta(self, dispositivo: EntidadeForte, porta: PortaInterface) -> None:
        if porta.numero_porta in dispositivo.portas_associadas:
            raise ValueError(
                f"[VIOLACAO DE DISCRIMINADOR] O dispositivo '{dispositivo.hostname}' "
                f"já possui a porta '{porta.numero_porta}' cadastrada!"
            )
        porta.dispositivo_proprietario = dispositivo
        dispositivo.portas_associadas[porta.numero_porta] = porta
        self.ligacoes[porta.chave_completa] = porta
        print(f"✓ Porta '{porta.numero_porta}' vinculada com sucesso a '{dispositivo.hostname}'.")

### 2. Executando o Teste de Entidade Fraca e Discriminador

In [ ]:
rel = RelacionamentoIdentificador()

rt1 = EntidadeForte("10.0.1.1", "RT-IFCE-TAUA", "Rack Central")
sw1 = EntidadeForte("10.0.1.2", "SW-CORE-TAUA", "Rack Central")

# Ambas as portas possuem o mesmo discriminador 'Gi0/1'
p1 = PortaInterface("Gi0/1", "Fibra Óptica", 1000)
p2 = PortaInterface("Gi0/1", "Par Trançado", 1000)

rel.vincular_porta(rt1, p1)
rel.vincular_porta(sw1, p2)

print(f"Chave Composta da Porta 1: {p1.chave_completa}")
print(f"Chave Composta da Porta 2: {p2.chave_completa}")

### 3. Modelo EER: Superclasse, Subclasses e Herança

In [ ]:
class DispositivoGenerico:
    def __init__(self, patrimonio: str, fabricante: str, modelo: str):
        self.patrimonio = patrimonio
        self.fabricante = fabricante
        self.modelo = modelo

class Roteador(DispositivoGenerico):
    def __init__(self, patrimonio: str, fabricante: str, modelo: str, tabela_rotas_max: int, suporta_bgp: bool):
        super().__init__(patrimonio, fabricante, modelo)
        self.tabela_rotas_max = tabela_rotas_max
        self.suporta_bgp = suporta_bgp

class Switch(DispositivoGenerico):
    def __init__(self, patrimonio: str, fabricante: str, modelo: str, quantidade_portas: int, capacidade_switching_gbps: float):
        super().__init__(patrimonio, fabricante, modelo)
        self.quantidade_portas = quantidade_portas
        self.capacidade_switching_gbps = capacidade_switching_gbps

r = Roteador("PAT-001", "Cisco", "ASR 1001-X", 1000000, True)
s = Switch("PAT-002", "Huawei", "S5735", 48, 176.0)

print("Herança em Roteador:", vars(r))
print("Herança em Switch:", vars(s))